# Conversations — Try it in PyTorch

An **optional** hands-on companion to [Chapter 11](https://learnai.robennals.org/conversations). The chapter says a chat is one long document with markers in it, and that the model keeps nothing between turns. Here you print that document and watch it grow.

New to PyTorch? The [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) is a quick introduction.

## About the models used here

The models in this notebook are small enough to run free, in your browser, in a few minutes. They are hundreds of times smaller than the ones behind ChatGPT or Claude, and it shows: they make mistakes a frontier model would not. What they do have is the same machinery. Everything here works the same way at a thousand times the size, which is the point.

In [ ]:
!pip install -q transformers accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE != "cpu" else torch.float32
print("running on", DEVICE)

chat_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
chat_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-1.7B", dtype=DTYPE).to(DEVICE)

def complete(tokenizer, model, text, max_new_tokens=60):
    """Give the model a piece of text and return what it writes next.

    The small limits here are about how much to print, not about how much the
    model is allowed to say.
    """
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("model loaded")

## What the model is really handed

The chapter says a chat is a prompt and a completion, with markers saying who is speaking. Here is the actual text.

In [ ]:
messages = [
    {"role": "user", "content": "What's the capital of Australia?"},
]

prompt_text = chat_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

print(repr(prompt_text))

Those `<|im_start|>` and `<|im_end|>` pieces are real tokens with their own numbers in the vocabulary, not punctuation the model has to interpret.

In [ ]:
for marker in ["<|im_start|>", "<|im_end|>"]:
    print(marker, "->", chat_tokenizer.encode(marker))

The prompt ends with the marker that opens the model's turn. That is what makes an answer the thing left to write.

## Repeated turns

The model keeps nothing between turns. Everything it needs has to be in the prompt, so the whole conversation goes back every time, including its own earlier replies.

In [ ]:
conversation = [{"role": "user", "content": "What's the capital of Australia?"}]

for turn in range(3):
    prompt_text = chat_tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    n_tokens = len(chat_tokenizer(prompt_text).input_ids)
    reply = complete(chat_tokenizer, chat_model, prompt_text, 40).strip()
    print(f"turn {turn + 1}: prompt is {n_tokens} tokens -> {reply[:70]!r}")
    conversation.append({"role": "assistant", "content": reply})
    conversation.append({"role": "user", "content": "Are you sure?"})

The prompt grows every turn. Nothing was remembered; it was re-read.

You may also spot the model getting a fact wrong somewhere in there. A 1.7B model does that. It is worth seeing rather than hiding: the machinery is the same as a frontier model's, and the size is what buys reliability.

Here is the third turn's prompt in full. The model's own first answer is in there, put back by the code above.

In [ ]:
print(chat_tokenizer.apply_chat_template(
    conversation[:5], tokenize=False, add_generation_prompt=True, enable_thinking=False))

## The system prompt

The chapter's last section adds one more message, at the top, that the human never sees. Nothing about the machinery changes: it is another message in the same stream, written by the program rather than by you.

In [ ]:
with_system = [
    {"role": "system", "content": "You are Otter, a friendly assistant who keeps "
                                  "answers short. Today is 29 August 2026. "
                                  "The user is in Bristol."},
    {"role": "user", "content": "What's the capital of Australia?"},
]

print(chat_tokenizer.apply_chat_template(
    with_system, tokenize=False, add_generation_prompt=True, enable_thinking=False))

Compare that with the prompt further up. The only difference is one more message at the front, wrapped in exactly the same markers as yours.

## What you saw

- A conversation is one stream of text with markers in it. The markers are ordinary tokens.
- Nothing is remembered between turns. The whole conversation is sent again each time, including the model's own earlier replies.
- The prompt therefore grows every turn, which is why the cost of a long conversation adds up.